# The Alpha Transform

## Stability Inversion for Fixed-Point Iteration

**Author:** Keenan M Stone  
**Origin:** May 2017 (original Petrification scripts)  
**This notebook:** April 2026  
**Status:** Self-contained review of the alpha transform, its theory, applications, and placement in the literature

---

### Purpose

This notebook develops the **alpha-transform** — the map

$$g(x) = \alpha f(x) + (1-\alpha)x$$

as a tool for controlling fixed-point stability. We:

1. State and prove the core theorems on stability inversion
2. Develop the position-dependent generalization $\alpha(x)$
3. Connect to the literature (Krasnoselskii-Mann, SOR, Anderson mixing)
4. Demonstrate applications on discrete maps (logistic, Migdal-Kadanoff)
5. Connect fixed points to eigenstates in projective Hilbert space
6. Assess what, if anything, is novel vs. well-known

The alpha transform likely fits into a broader class of **concavity-inverting transforms** — it modifies the curvature landscape of an iteration map to swap attractors and repulsors. Whether this framing adds value beyond existing theory is an open question we address honestly.

---

### Companion Notebooks

This notebook develops the core theory. The ideas are explored and applied in several companion notebooks:

| Notebook | Focus | Key result |
|----------|-------|------------|
| **[eigenstate_fixedpoint_correspondence](eigenstate_fixedpoint_correspondence.ipynb)** | Full investigation: eigenstates as fixed points, Riccati-Bloch, Frobenius-Perron | Rigorous dead-end analysis; Koopman connection |
| **[turbiner_riccati_bloch](turbiner_riccati_bloch.ipynb)** [§6 of eigenstate] | Turbiner's nonlinearization in Riccati form | Eigenvalues ↔ bounded Riccati trajectories |
| **[crossover_alpha_turbiner](crossover_alpha_turbiner.ipynb)** | α-relaxation applied to the Dyson equation | **Novel**: $\alpha^* \approx 0.5$ universality for Green's functions |
| **[inverse_correspondence](inverse_correspondence.ipynb)** | Regular vs. inverted ($\alpha = -1$) dynamics | Lyapunov comparison (open); OGY-like chaos control via $\alpha(x)$ |
| **[perturbation_detection](perturbation_detection.ipynb)** | Classical: deducing perturbation forces from $\alpha(x)$ | Exact reconstruction: $V'_\text{pert} = kx(\alpha(x) - 1)$ |
| **[quantum_perturbation_detection](quantum_perturbation_detection.ipynb)** | Quantum: hydrogen atom perturbation stacking | Shape of $\alpha(r)-1$ classifies perturbation type |

## 1. Literature Review: Fixed Points and Eigenstates

### 1.1 Fixed Points in Classical Dynamical Systems

The theory of fixed-point iteration is foundational to analysis. A map $f: X \to X$ on a metric space has a fixed point $x^*$ where $f(x^*) = x^*$. The central results:

- **Banach contraction mapping theorem** (1922): If $f$ is a contraction ($|f(x) - f(y)| \le L|x-y|$ with $L < 1$), the iteration $x_{n+1} = f(x_n)$ converges to a unique fixed point from any initial condition.
- **Local stability**: For $C^1$ maps on $\mathbb{R}$, a fixed point is stable (attracting) if $|f'(x^*)| < 1$ and unstable (repelling) if $|f'(x^*)| > 1$.
- **Bifurcation theory**: As parameters vary, fixed points can change stability, split, or disappear. The period-doubling cascade in the logistic map $f(x) = ax(1-x)$ is the prototypical example [Strogatz, Ch. 10].

### 1.2 Fixed Points in Quantum Mechanics — The Projective Perspective

The eigenvalue equation $\hat{A}|\psi\rangle = \lambda|\psi\rangle$ says: an operator acts on a state and returns *the same state* up to scaling. In **projective Hilbert space** $\mathbb{P}(\mathcal{H})$ — where $|\psi\rangle \sim c|\psi\rangle$ — this is exactly a **fixed point** of the projective action:

$$\Pi_A : [|\psi\rangle] \mapsto \left[\frac{\hat{A}|\psi\rangle}{\|\hat{A}|\psi\rangle\|}\right]$$

Then $[|\psi_k\rangle]$ is a fixed point of $\Pi_A$ if and only if $|\psi_k\rangle$ is an eigenstate of $\hat{A}$.

| Fixed-point dynamics | Quantum eigenvalue problem |
|---|---|
| $f(x^*) = x^*$ | $\hat{A}\vert\psi\rangle = \lambda\vert\psi\rangle$ |
| $\lvert f'(x^*)\rvert < 1$: stable (attracting) | $\lvert\lambda\rvert$ largest: power method converges |
| $\lvert f'(x^*)\rvert > 1$: unstable (repelling) | $\lvert\lambda\rvert$ not largest: power method diverges |
| Alpha-transform $g = \alpha f + (1-\alpha)x$ | Relaxed iteration / spectral shift |
| Bifurcation in parameter $a$ | Spectral phase transitions |

**This is not just an analogy — it is the mathematical content of power iteration** [Trefethen & Bau, Ch. 25].

Several concrete algorithms already embody this connection:

- **Power iteration** converges to the eigenstate with largest $\lvert\lambda\rvert$ — the most "stable" fixed point of $\Pi_A$
- **Inverse power iteration with shift** $(\hat{A} - \sigma I)^{-1}$ converges to the eigenstate *nearest* $\sigma$ — fixed-point iteration with a *modified stability landscape*
- **Spectral shooting methods** find eigenvalues as zeros of a boundary mismatch — root-finding that is itself a fixed-point problem

### 1.3 Renormalization Group as Fixed-Point Iteration

Wilson's renormalization group [Wilson 1975] treats phase transitions as fixed points of block-spin maps. Migdal-Kadanoff approximations [Migdal 1975, Kadanoff 1966] reduce lattice gauge theory to iterated maps. The RG flow $\beta$-function is a continuous-time analog of what we study here discretely.

### 1.4 Relaxation Methods in Numerical Analysis

The alpha-transform is closely related to established techniques:

- **Krasnoselskii-Mann iteration** [Krasnoselskii 1955]: $x_{n+1} = (1-\theta)x_n + \theta T(x_n)$ for nonexpansive $T$. Setting $\theta = \alpha$ recovers our transform.
- **Successive Over-Relaxation (SOR)**: In numerical linear algebra, the SOR method uses a mixing parameter $\omega$ to accelerate Gauss-Seidel iteration. The optimal $\omega$ depends on the spectral radius of the iteration matrix.
- **Anderson mixing / DIIS**: More sophisticated self-consistent-field methods that use a history of iterates to extrapolate. These are nonlinear generalizations.

**The honest question:** Is the alpha-transform anything more than a relabeling of these known techniques? The answer may be that the value lies not in the transform itself but in the systematic framework for choosing $\alpha$ — particularly the position-dependent $\alpha(x)$ generalization.

### 1.5 References

1. **Strogatz, S.H.** *Nonlinear Dynamics and Chaos.* 2nd ed., CRC Press, 2015.
2. **Trefethen, L.N. and Bau, D.** *Numerical Linear Algebra.* SIAM, 1997.
3. **Wilson, K.G.** "The renormalization group." *Rev. Mod. Phys.* 47.4 (1975): 773.
4. **Kadanoff, L.P.** "Scaling laws for Ising models near $T_c$." *Physics* 2.6 (1966): 263.
5. **Migdal, A.A.** "Recursion equations in gauge theories." *Sov. Phys. JETP* 42.3 (1975): 413.
6. **Krasnoselskii, M.A.** "Two remarks on the method of successive approximations." *Uspekhi Mat. Nauk* 10.1 (1955): 123.
7. **Güttel, S. and Tisseur, F.** "The nonlinear eigenvalue problem." *Acta Numerica* 26 (2017): 1–94.
8. **Stone, K.M.** "Petrification scripts." (2017, unpublished).

## 2. Fixed-Point Analysis: The On-Mapping Procedure

Before introducing the alpha transform, we need the visual framework that led to its discovery.

### 2.1 Rendering a Map on the Bisector

To study the fixed-point behavior of a map $f: \mathbb{R} \to \mathbb{R}$, we plot two things:

1. The **graph** of $f(x)$ — the curve $y = f(x)$
2. The **bisector** — the line $y = x$

Fixed points are the intersections: $f(x^*) = x^* \Leftrightarrow$ the curve crosses the bisector. This is the standard **cobweb diagram** setup, where we can visually trace the iteration $x_{n+1} = f(x_n)$ by bouncing between the curve and the bisector.

The bisector is doing real work here — it's not just a reference line. It's the **identity map** $\text{id}(x) = x$, and fixed points are exactly the places where $f$ and $\text{id}$ agree. The cobweb procedure works because we're alternating between "apply $f$" (move vertically to the curve) and "read off the input" (move horizontally to the bisector).

This "on-mapping" — rendering the function relative to the bisector so that the fixed-point structure becomes geometric — is what makes cobweb analysis possible. Everything that follows comes from taking this geometric view seriously.

### 2.2 The Slope at the Crossing

At each intersection, the **angle** between the curve and the bisector determines the dynamics:

- If the curve crosses the bisector with $|f'(x^*)| < 1$, the cobweb spirals **inward** — the fixed point is **attracting** (stable)
- If $|f'(x^*)| > 1$, the cobweb spirals **outward** — the fixed point is **repelling** (unstable)
- If $f'(x^*) < 0$, the cobweb oscillates (alternating sides); if $f'(x^*) > 0$, it approaches monotonically

The slope relative to the bisector is what matters: $f'(x^*) - 1$ measures how steeply the curve departs from $y = x$ at the fixed point. This observation is the seed of the alpha transform.

## 3. Visual Discovery of the Alpha Transform

The alpha transform was discovered by taking the on-mapping view literally and asking: *what happens if we subtract the bisector, scale, and add it back?*

### 3.1 Step 1: Subtract the Bisector

Define the **deviation function**:

$$h(x) = f(x) - x$$

This shifts the picture so that all fixed points now sit on the x-axis: $h(x^*) = f(x^*) - x^* = 0$. The fixed points of $f$ are the **roots** of $h$.

This is a simple coordinate change, but it reveals something powerful: the roots of $h$ are at $h = 0$, so **multiplying $h$ by any nonzero scalar $\alpha$ cannot move them**. The zeros stay zeros. What changes is the slope — and the slope at a zero of $h$ is exactly $h'(x^*) = f'(x^*) - 1$, which is the quantity that controls stability.

### 3.2 Step 2: Scale the Deviation

Multiply $h$ by a constant $\alpha$:

$$\alpha \cdot h(x) = \alpha(f(x) - x)$$

The roots haven't moved. But the slope at each root has been rescaled from $f'(x^*) - 1$ to $\alpha(f'(x^*) - 1)$. We can make slopes steeper or shallower, and we can flip their sign.

### 3.3 Step 3: Add the Bisector Back

To get a proper iteration map, add back $y = x$:

$$g(x) = \alpha(f(x) - x) + x = \alpha f(x) + (1 - \alpha)x$$

This is the **alpha-transformed map**. It has the same fixed points as $f$, but the slope at each fixed point is now:

$$g'(x^*) = \alpha f'(x^*) + (1 - \alpha) = 1 + \alpha(f'(x^*) - 1)$$

The three-step procedure (subtract bisector → scale → add bisector back) makes the transform completely transparent:

### 3.4 What Different $\alpha$ Values Do

**$\alpha > 1$** (amplify the deviation): The curve is pulled **away** from the bisector. Slopes at fixed points become more extreme. A mildly unstable point ($f' = 1.2$) becomes more unstable ($g' = 1 + \alpha \cdot 0.2$). Stable period-1 orbits can bifurcate into period-2, period-4, and ultimately chaos — around the same fixed-point positions. This lets you **probe more extreme versions of the system** without changing where the fixed points are.

**$0 < \alpha < 1$** (damp the deviation): The curve is pulled **toward** the bisector. All slopes are compressed toward 1. Bifurcations relax — chaotic regimes simplify back to periodic, and periodic orbits collapse to fixed points. This lets you **inspect the actual roots** that are hidden behind chaotic behavior in the original system.

**$\alpha < 0$** (flip the deviation): The curve is reflected **through** the bisector. This is the key insight: **stability and instability switch**. A repulsive fixed point ($|f'| > 1$) becomes attractive ($|g'| < 1$), and vice versa. Now the standard cobweb iteration converges to what were previously the *repulsive* fixed points.

This lets you use the same fixed-point analysis procedure to **inspect unstable fixed points directly** — points that the original iteration flees from. As other parameters of the system are tuned, these apparent repulsors move through phase space, and their trajectories reveal structure (bifurcations of the unstable manifold) that is invisible in the standard forward iteration.

### 3.5 Why This Matters: Seeing Both Sides of the Bifurcation

In the logistic map, the standard bifurcation diagram shows the **stable** attractors: period-1 → period-2 → period-4 → chaos as $a$ increases. But the unstable fixed points are also bifurcating — they just don't show up in forward iteration because trajectories flee them.

With $\alpha < 0$, we can construct the **unstable bifurcation diagram** — the hidden structure where repulsors split and move. This opens several questions:

1. **Can we understand chaos better by tracking both stable and unstable fixed points simultaneously?** The interaction between approaching attractors and retreating repulsors may reveal the mechanism of bifurcation more clearly than tracking either alone.

2. **Can discrepancies between stable and unstable bifurcation structures reveal missing physics?** If experimental data shows bifurcation at a parameter value where the model's unstable fixed points aren't in the right place, this could indicate an unaccounted-for influence — a hidden variable, a neglected coupling, or an incorrect functional form near those roots.

3. **Can we work backwards from the combined (stable + unstable) dynamics to reconstruct unmodeled effects?** Knowing *how* the unstable fixed points are misbehaving near the roots of the original system constrains what kind of perturbation could cause that misbehavior.

These are open questions. The alpha transform provides the tool to investigate them — the rest of this notebook develops the formal theory and computational demonstrations.

In [ ]:
# Visual walkthrough: subtract bisector, scale, add back
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from petrification.maps import logistic

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 8), 'font.size': 12})

a = 3.5  # chaotic-ish logistic
x = np.linspace(-0.1, 1.1, 500)
fx = logistic(a, x)

# Fixed points
x_fp0, x_fp1 = 0.0, 1 - 1/a  # 0 and (a-1)/a

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# ── Row 1: The three-step procedure ──

# Step 1: f(x) on the bisector
ax = axes[0, 0]
ax.plot(x, fx, 'b-', lw=2, label=r'$f(x) = ax(1-x)$')
ax.plot(x, x, 'k-', lw=1, label=r'$y = x$ (bisector)')
ax.plot(x_fp0, x_fp0, 'ro', ms=10, zorder=5)
ax.plot(x_fp1, x_fp1, 'ro', ms=10, zorder=5)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title('Step 0: On-mapping\nFixed points where $f$ crosses bisector')
ax.legend(fontsize=9); ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.5, 1.2)
ax.set_aspect('equal')

# Step 2: Subtract bisector → h(x) = f(x) - x
hx = fx - x
ax = axes[0, 1]
ax.plot(x, hx, 'b-', lw=2, label=r'$h(x) = f(x) - x$')
ax.axhline(0, color='k', lw=1)
ax.plot(x_fp0, 0, 'ro', ms=10, zorder=5)
ax.plot(x_fp1, 0, 'ro', ms=10, zorder=5)
ax.set_xlabel('$x$'); ax.set_ylabel('$h(x)$')
ax.set_title('Step 1: Subtract bisector\nFixed points now at $h = 0$')
ax.legend(fontsize=9); ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.5, 1.2)

# Step 3: Scale h(x) by different α
ax = axes[0, 2]
for alpha_v, color, ls in [(1.0, 'blue', '-'), (0.5, 'green', '--'),
                             (1.5, 'orange', '-.'), (-1.0, 'red', ':')]:
    ax.plot(x, alpha_v * hx, color=color, lw=2, ls=ls,
            label=rf'$\alpha = {alpha_v}$')
ax.axhline(0, color='k', lw=1)
ax.plot(x_fp0, 0, 'ro', ms=10, zorder=5)
ax.plot(x_fp1, 0, 'ro', ms=10, zorder=5)
ax.set_xlabel('$x$'); ax.set_ylabel(r'$\alpha \cdot h(x)$')
ax.set_title('Step 2: Scale deviation\nRoots unchanged, slopes rescaled')
ax.legend(fontsize=9); ax.set_xlim(-0.15, 1.15); ax.set_ylim(-1.5, 1.5)

# ── Row 2: Add bisector back — the resulting iteration maps ──

alphas_demo = [0.5, 1.0, 1.5, -0.5]
titles = [r'$\alpha = 0.5$: damped' + '\nBifurcation relaxed',
          r'$\alpha = 1.0$: original',
          r'$\alpha = 1.5$: amplified' + '\nMore extreme dynamics',
          r'$\alpha = -0.5$: flipped' + '\nStability inverted']
colors_demo = ['green', 'blue', 'orange', 'red']

for idx, (alpha_v, title, color) in enumerate(zip(alphas_demo, titles, colors_demo)):
    if idx < 3:
        ax = axes[1, idx]
    else:
        # Use inset or overlay on last panel — but we have 3 cols only.
        # Put the flip case on the 3rd panel as a second curve
        ax = axes[1, 2]

    gx = alpha_v * fx + (1 - alpha_v) * x
    if idx < 3:
        ax.plot(x, gx, color=color, lw=2,
                label=rf'$g(x) = \alpha f + (1-\alpha)x$')
        ax.plot(x, x, 'k-', lw=1)
        ax.plot(x_fp0, x_fp0, 'ro', ms=8, zorder=5)
        ax.plot(x_fp1, x_fp1, 'ro', ms=8, zorder=5)
        # Show slope
        lam1 = a * (1 - 2*x_fp1)
        g_prime = 1 + alpha_v * (lam1 - 1)
        ax.set_xlabel('$x$'); ax.set_ylabel('$g(x)$')
        ax.set_title(title)
        ax.text(0.02, 0.02,
                f"$g'(x^*_1) = {g_prime:.2f}$\n"
                f"{'stable' if abs(g_prime) < 1 else 'unstable'}",
                transform=ax.transAxes, fontsize=10,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.5, 1.3)
        ax.legend(fontsize=8)

# Redo panel [1,2] to show the flipped case
ax = axes[1, 2]
ax.cla()
alpha_flip = -0.5
gx_flip = alpha_flip * fx + (1 - alpha_flip) * x
ax.plot(x, gx_flip, 'r-', lw=2, label=rf'$g(x)$, $\alpha = {alpha_flip}$')
ax.plot(x, x, 'k-', lw=1)
ax.plot(x_fp0, x_fp0, 'ro', ms=8, zorder=5)
ax.plot(x_fp1, x_fp1, 'ro', ms=8, zorder=5)
lam0, lam1 = a, a*(1-2*x_fp1)
g0 = 1 + alpha_flip * (lam0 - 1)
g1 = 1 + alpha_flip * (lam1 - 1)
ax.set_xlabel('$x$'); ax.set_ylabel('$g(x)$')
ax.set_title(r'$\alpha = -0.5$: flipped' + '\nStability inverted')
ax.text(0.02, 0.02,
        f"$g'(0) = {g0:.2f}$ ({'stable' if abs(g0) < 1 else 'unstable'})\n"
        f"$g'(x^*_1) = {g1:.2f}$ ({'stable' if abs(g1) < 1 else 'unstable'})",
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.5, 1.5)
ax.legend(fontsize=8)

plt.suptitle(f'The Alpha Transform: Subtract Bisector → Scale → Add Back (logistic, $a = {a}$)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the stability analysis
print(f"\nLogistic map at a = {a}:")
print(f"  x*₀ = {x_fp0:.4f}, f'(x*₀) = {lam0:+.2f} (unstable)")
print(f"  x*₁ = {x_fp1:.4f}, f'(x*₁) = {lam1:+.2f} "
      f"({'stable' if abs(lam1) < 1 else 'unstable'})")
print(f"\nAfter alpha transform:")
for alpha_v in [0.5, 1.0, 1.5, -0.5, -1.0]:
    g0 = 1 + alpha_v * (lam0 - 1)
    g1 = 1 + alpha_v * (lam1 - 1)
    print(f"  α={alpha_v:+5.1f}: g'(x*₀)={g0:+6.2f} "
          f"({'✓' if abs(g0) < 1 else '✗'}), "
          f"g'(x*₁)={g1:+6.2f} "
          f"({'✓' if abs(g1) < 1 else '✗'})")

In [ ]:
# Cobweb: iterating to the UNSTABLE fixed point via negative α
from petrification.iteration import cobweb_data
from petrification.bifurcation import compute_bifurcation, compute_bifurcation_transformed

a_demo = 4.0
x0_demo = 0.1
x_fp_unstable = 0.0  # f'(0) = 4, repelling in original
x_fp_stable = 1 - 1/a_demo  # f'(0.75) = -2, also unstable at a=4

# α that stabilizes x*=0 (needs α < 0 since f'(0) = 4 > 1)
alpha_stab_0 = 1.0 / (1.0 - a_demo)  # = -1/3

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

# Panel 1: Original — cobweb flies away from x*=0
Ix, Iy = cobweb_data(logistic, a_demo, x0_demo, n_iter=15)
lx = np.linspace(-0.1, 1.1, 500)

ax = axes[0]
ax.plot(lx, logistic(a_demo, lx), 'b-', lw=2)
ax.plot(lx, lx, 'k-', lw=1)
ax.plot(Ix, Iy, 'r-', lw=0.8, alpha=0.8)
ax.plot(x_fp_unstable, x_fp_unstable, 'rx', ms=12, mew=3, zorder=5,
        label=f'$x^*=0$ (repeller, $f\'={a_demo:.0f}$)')
ax.set_title(f'Original ($\\alpha = 1$)\nCobweb flees $x^*=0$')
ax.set_xlabel('$x_n$'); ax.set_ylabel('$x_{n+1}$')
ax.legend(fontsize=8); ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.15, 1.15)

# Panel 2: α = -1/3 — cobweb converges TO x*=0
Ix_t, Iy_t = cobweb_data(logistic, a_demo, x0_demo, n_iter=40, alpha=alpha_stab_0)
gx = alpha_stab_0 * logistic(a_demo, lx) + (1 - alpha_stab_0) * lx

ax = axes[1]
ax.plot(lx, gx, 'r-', lw=2)
ax.plot(lx, lx, 'k-', lw=1)
ax.plot(Ix_t, Iy_t, 'g-', lw=0.8, alpha=0.8)
ax.plot(x_fp_unstable, x_fp_unstable, 'go', ms=10, zorder=5,
        label=f'$x^*=0$ (now attractor, $g\'={1 + alpha_stab_0*(a_demo-1):.2f}$)')
ax.set_title(f'$\\alpha = {alpha_stab_0:.3f}$\nCobweb converges to former repeller')
ax.set_xlabel('$x_n$'); ax.set_ylabel('$x_{n+1}$')
ax.legend(fontsize=8); ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.5, 1.5)

# Panel 3: Dual bifurcation diagram — stable AND unstable structures
a_range = np.linspace(2.5, 4.0, 400)
x0_samples = np.linspace(0.01, 0.99, 30)

# Standard (shows stable attractors)
a_s, x_s = compute_bifurcation(logistic, a_range, x0_samples,
                                n_iter=500, n_discard=400)
# Flipped α = -1 (shows unstable structure)
a_u, x_u = compute_bifurcation_transformed(logistic, -1.0, a_range,
                                             x0_samples, n_iter=500, n_discard=400)

ax = axes[2]
ax.scatter(a_s, x_s, s=0.1, c='steelblue', alpha=0.4, label='$\\alpha=1$ (stable attractors)')
ax.scatter(a_u, x_u, s=0.1, c='red', alpha=0.3, label='$\\alpha=-1$ (unstable "attractors")')
ax.set_xlabel('Parameter $a$'); ax.set_ylabel('$x$')
ax.set_title('Stable + Unstable Bifurcation Structure\nBoth diagrams share the same fixed points')
ax.legend(fontsize=8, loc='upper left')

plt.suptitle('Negative α reveals the hidden dynamics of repulsive fixed points',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The blue structure shows where orbits go (standard bifurcation diagram).")
print("The red structure shows where orbits flee FROM — the unstable skeleton.")
print("Together they give a more complete picture of the system's phase-space geometry.")

## 4. The Alpha Transform: Formal Theory

The visual intuition from §3 (subtract bisector → scale → add back) now gets precise statements and proofs.

### 4.1 Definition and Fixed-Point Preservation

Let $f: \mathbb{R} \to \mathbb{R}$ be a $C^1$ map with fixed points $\{x_i^*\}$ where $f(x_i^*) = x_i^*$. Define:

$$g(x) = \alpha f(x) + (1 - \alpha) x$$

Since $f(x_i^*) = x_i^*$, we have $g(x_i^*) = \alpha x_i^* + (1-\alpha) x_i^* = x_i^*$, so **$g$ has exactly the same fixed points as $f$** for any $\alpha$.

### 4.2 Proofs

**Proposition 1 (Transformed slope).** At any fixed point $x^*$:
$$g'(x^*) = \alpha f'(x^*) + (1 - \alpha) = 1 + \alpha\bigl(f'(x^*) - 1\bigr)$$

*Proof.* $g'(x) = \alpha f'(x) + (1 - \alpha)$. Evaluate at $x = x^*$. $\square$

Writing $\lambda = f'(x^*)$, the transformed slope is $g'(x^*) = 1 + \alpha(\lambda - 1)$.

---

**Proposition 2 (Stability interval).** The fixed point $x^*$ is stable under $g$ (i.e., $|g'(x^*)| < 1$) if and only if:
$$-\frac{2}{\lambda - 1} < \alpha < 0 \quad \text{when } \lambda > 1$$
$$0 < \alpha < \frac{2}{1 - \lambda} \quad \text{when } \lambda < -1$$

*Proof.* We need $|1 + \alpha(\lambda-1)| < 1$. Expanding:
$$-1 < 1 + \alpha(\lambda - 1) < 1$$
$$-2 < \alpha(\lambda - 1) < 0$$
When $\lambda > 1$: $\lambda - 1 > 0$, so $-2/(\lambda-1) < \alpha < 0$.  
When $\lambda < -1$: $\lambda - 1 < 0$, dividing reverses inequalities, giving $0 < \alpha < 2/(1-\lambda)$. $\square$

---

**Proposition 3 (Optimal alpha).** The value $\alpha^* = 1/(1 - \lambda)$ **zeroes the transformed slope** ($g'(x^*) = 0$), giving the fastest possible local convergence.

*Proof.* $g'(x^*) = 1 + \alpha(\lambda - 1) = 1 + \frac{\lambda - 1}{1 - \lambda} = 1 - 1 = 0$. $\square$

Note: $\alpha^*$ is the midpoint of the stability interval.

---

**Proposition 4 (Obstruction to global inversion).** If $f$ has both an unstable fixed point with $\lambda_i > 1$ and another with $\lambda_j < -1$, then **no single $\alpha$ can stabilize both simultaneously**.

*Proof.* Stabilizing $\lambda_i > 1$ requires $\alpha < 0$. Stabilizing $\lambda_j < -1$ requires $\alpha > 0$. These are disjoint. $\square$

**Example:** For the logistic map at $a = 4$, the fixed points are $x_0^* = 0$ with $f'(0) = 4 > 1$, and $x_1^* = 3/4$ with $f'(3/4) = -2 < -1$. No single alpha stabilizes both.

---

**Proposition 5 (Same-sign stabilization).** If all unstable fixed points have slopes of the same sign (all $\lambda_i > 1$ or all $\lambda_i < -1$), then $\alpha^* = 1/(1 - \lambda_{\max})$ stabilizes **all** of them simultaneously, and furthermore **destabilizes all previously stable fixed points**.

*Proof.* Consider the case $\lambda_i > 1$ for all unstable $x_i^*$. Take $\lambda_{\max} = \max_i \lambda_i$ and $\alpha^* = 1/(1-\lambda_{\max}) < 0$.

For any unstable $x_i^*$: Since $1 \leq \lambda_i \leq \lambda_{\max}$, we have $0 \leq (\lambda_i-1)/(\lambda_{\max}-1) \leq 1$, so $g'(x_i^*) = 1 - (\lambda_i - 1)/(\lambda_{\max} - 1) \in [0, 1) \subset (-1,1)$. Stable. ✓

For any stable $x_j^*$ with $|\lambda_j| < 1$: Since $\lambda_j < 1$ and $1-\lambda_{\max} < 0$, the correction term is positive: $g'(x_j^*) > 1$. Unstable. ✓ $\square$

### 4.3 Connection to the Visual Intuition

Each proposition has a direct geometric interpretation from the subtract-scale-add picture:

- **Prop 1:** Scaling the deviation $h = f - \text{id}$ by $\alpha$ rescales the departure from the bisector → the slope at each zero of $h$ gets affinely transformed.
- **Prop 2:** The stability interval is where the scaled curve stays close enough to the bisector that the cobweb spirals inward.
- **Prop 3:** Optimal $\alpha$ makes the curve **tangent** to the bisector at $x^*$ — the cobweb collapses in one step (locally).
- **Prop 4:** When different fixed points need the deviation flipped in opposite directions, no single scaling can do both.
- **Prop 5:** When all instabilities point the same way, one flip handles them all — but necessarily ejects the previously stable points.

### 4.4 Connection to Curvature

To find $\lambda_{\max} = \max_x f'(x)$, we solve $f''(x) = 0$ for critical points of $f'$. For polynomials, these are inflection points, not points of maximum curvature $|f''|$. The coincidence for simple maps (the logistic map has constant $f''$) may have caused earlier confusion between "inverse of slope at max curvature" and the correct formula.

### 4.5 Operator Structure

Using operator notation $\Gamma_\alpha f(x) = \alpha f(x) + (1-\alpha)x$:
- $\Gamma_1 = \text{id}$ (identity)
- $\Gamma_\alpha \Gamma_\beta = \Gamma_{\alpha\beta}$ (successive applications compose multiplicatively)
- $\Gamma_{\alpha^{-1}} = \Gamma_\alpha^{-1}$ (invertible)
- Isomorphic to $(\mathbb{R}_{>0}, \cdot)$ under $\alpha \mapsto \Gamma_\alpha$

## 5. The Position-Dependent Generalization: $\alpha(x)$

### 5.1 Theory

Consider $\alpha(x)$:
$$g(x) = \alpha(x) f(x) + (1 - \alpha(x)) x$$

**Fixed points are still preserved:** if $f(x^*) = x^*$, then $g(x^*) = \alpha(x^*) x^* + (1-\alpha(x^*)) x^* = x^*$ for any $\alpha(x^*)$.

**Derivative at a fixed point:**
$$g'(x^*) = \alpha'(x^*)\underbrace{(f(x^*) - x^*)}_{= 0} + \alpha(x^*) f'(x^*) + (1 - \alpha(x^*)) = 1 + \alpha(x^*)(f'(x^*) - 1)$$

The $\alpha'$ term vanishes because $f(x^*) = x^*$. So **the stability at each fixed point depends only on $\alpha(x^*)$**, not on $\alpha'(x^*)$.

### 5.2 Overcoming Proposition 4

The obstruction was that no single constant $\alpha$ can stabilize fixed points with slopes of both signs. With $\alpha(x)$, each fixed point gets its own $\alpha$ value — the choice $\alpha(x^*) = 1/(1 - f'(x^*))$ zeroes $g'$ at $x^*$ automatically, for each fixed point independently.

### 5.3 Gauge Structure

With position-dependent $\alpha(x)$, the global group structure ($\Gamma_\alpha \Gamma_\beta = \Gamma_{\alpha\beta}$) becomes a **gauge transformation** — the parameter varies over the base space, analogous to how gauge fields in physics assign different group elements to different spacetime points.

### 5.4 The Tradeoff

The formula $\alpha(x) = 1/(1-f'(a,x))$ has a singularity where $f'(a,x) = 1$, creating a "wall" in the iteration landscape. Trajectories passing near this singularity get disrupted. Capping $|\alpha| \leq C$ prevents divergence but fragments the basin of attraction. This means:

- **$\alpha(x)$ is locally superior** (superlinear convergence near each fixed point)
- **$\alpha(x)$ is globally inferior** (fragmented basin compared to constant $\alpha$)

This tradeoff is the central practical limitation.

## 6. Placing in the Literature: Concavity-Inverting Transforms

The alpha-transform alters the curvature landscape of an iteration map. At its core:

$$g''(x) = \alpha f''(x)$$

so for $\alpha < 0$, the concavity of $g$ is flipped relative to $f$. This inverts the relationship between curvature and fixed-point stability: what was concave-down (creating a stable attracting fixed point with $f' < 0$) becomes concave-up (repelling), and vice versa.

**How this relates to known techniques:**

| Technique | What it does | $\alpha$-transform equivalent |
|-----------|-------------|------------------------------|
| SOR ($\omega$-relaxation) | Mixes update with current value | Constant $\alpha$ |
| Krasnoselskii-Mann | $(1-\theta)x + \theta Tx$ | $\alpha = \theta$ |
| Anderson mixing | Uses history to extrapolate | $\alpha$ from past iterates |
| Broyden mixing (DMFT) | Quasi-Newton with rank-1 updates | Adaptive matrix $\alpha$ |
| **$\alpha(x)$ transform** | Position-dependent relaxation | Independent per fixed point |

**What the $\alpha(x)$ generalization adds beyond Krasnoselskii-Mann:**

1. The key result that $g'(x^*) = 1 + \alpha(x^*)(f'(x^*) - 1)$ — where the $\alpha'$ term vanishes — means position-dependent relaxation gives **independent local control** at each fixed point. This is a clean result that is implicitly known in the preconditioning literature but rarely stated in this explicit form.

2. The obstruction theorem (Prop. 4) and its resolution via $\alpha(x)$ is a useful pedagogical contribution.

3. The gauge-theoretic framing ($\alpha(x)$ as a gauge parameter varying over phase space) is suggestive but currently just language, not yet a source of new results.

**Honest assessment:** The alpha-transform is likely not a "new discovery" — the ideas are scattered across the fixed-point theory, numerical analysis, and preconditioning literature. The potential contribution is the **systematic synthesis**: stating the stability inversion theorems cleanly, proving the $\alpha'$ cancellation for position-dependent $\alpha$, identifying the obstruction and its resolution, and connecting all of this to eigenvalue problems via the projective fixed-point perspective.

## 7. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

from petrification.maps import logistic, migdal_kadanoff
from petrification.transforms import (alpha_transform, compute_optimal_alpha,
                                       make_alpha_func, optimal_alpha_at_x)
from petrification.bifurcation import compute_bifurcation, compute_bifurcation_transformed
from petrification.iteration import iterate, iterate_transformed, cobweb_data, find_fixed_points
from petrification.potentials import harmonic
from petrification.quantum import (
    discretize_hamiltonian, solve_eigenstates,
    inverse_power_iteration, projective_power_iteration, alpha_eigenstate_scan,
)

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 8), 'font.size': 12})
print('All imports successful.')

## 8. Demonstrations on Discrete Maps

### 8.1 Cobweb Diagrams: Stabilizing the Logistic Map

For the logistic map $f(x) = ax(1-x)$ at $a = 4.0$, the fixed point at $x^* = 3/4$ is **unstable** ($|f'(x^*)| = |2 - a| = 2 > 1$). The alpha-transform stabilizes it.

In [ ]:
a = 4.0
x0 = 0.7

# Compute optimal alpha
alpha, info = compute_optimal_alpha(logistic, a, x_domain=(0.0, 1.0))
print(f"Optimal alpha = {alpha:.4f}")
print(f"  Method: {info['method']}, df_max = {info['df_max']}, df_min = {info['df_min']}")

# Cobweb data
Ix_orig, Iy_orig = cobweb_data(logistic, a, x0, n_iter=20)
Ix_trans, Iy_trans = cobweb_data(logistic, a, x0, n_iter=20, alpha=alpha)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
lx = np.linspace(-0.1, 1.1, 500)

ax1.plot(lx, logistic(a, lx), 'b-', lw=2, label=r'$f(x) = ax(1-x)$')
ax1.plot(lx, lx, 'k-', lw=1)
ax1.plot(Ix_orig, Iy_orig, 'r-', lw=0.8, alpha=0.7)
ax1.set_xlabel(r'$x_n$'); ax1.set_ylabel(r'$x_{n+1}$')
ax1.set_title(f'Original: $a={a}$, diverges from fixed point')
ax1.set_xlim(-0.1, 1.1); ax1.set_ylim(-0.1, 1.1)
ax1.legend()

ly_t = alpha * logistic(a, lx) + (1 - alpha) * lx
ax2.plot(lx, ly_t, 'b-', lw=2, label=rf'$g(x) = \alpha f + (1-\alpha)x$')
ax2.plot(lx, lx, 'k-', lw=1)
ax2.plot(Ix_trans, Iy_trans, 'g-', lw=0.8, alpha=0.7)
ax2.set_xlabel(r'$x_n$'); ax2.set_ylabel(r'$x_{n+1}$')
ax2.set_title(rf'Alpha-transformed: $\alpha={alpha:.3f}$, converges to fixed point')
ax2.set_xlim(-0.1, 1.1); ax2.set_ylim(-0.1, 1.1)
ax2.legend()

plt.tight_layout()
plt.show()

### 8.2 Numerical Verification of the Stability Theorems

In [ ]:
print("=" * 72)
print("VERIFICATION: Alpha-Transform Stability for Logistic Map f(x)=ax(1-x)")
print("=" * 72)

for a_test in [2.5, 3.2, 3.83, 4.0]:
    x_fp = [0.0, 1 - 1/a_test]
    slopes = [a_test, 2 - a_test]
    
    print(f"\n--- a = {a_test} ---")
    for x_star, lam in zip(x_fp, slopes):
        alpha_opt = 1.0 / (1.0 - lam) if lam != 1 else float('inf')
        alpha_bdy = 2.0 / (1.0 - lam) if lam != 1 else float('inf')
        g_prime_opt = 1 + alpha_opt * (lam - 1) if np.isfinite(alpha_opt) else float('inf')
        g_prime_bdy = 1 + alpha_bdy * (lam - 1) if np.isfinite(alpha_bdy) else float('inf')
        stable_orig = "|\u03bb|<1 \u2713" if abs(lam) < 1 else "|\u03bb|\u22651 \u2717"
        
        print(f"  x* = {x_star:.4f}, f'(x*) = {lam:+.2f}  {stable_orig}")
        print(f"    \u03b1_optimal = {alpha_opt:+.4f} \u2192 g'(x*) = {g_prime_opt:+.4f}")
        print(f"    \u03b1_boundary = {alpha_bdy:+.4f} \u2192 g'(x*) = {g_prime_bdy:+.4f}")

print("\n" + "=" * 72)
print("Proposition 4 check: at a=4, x=0 needs \u03b1<0, x=3/4 needs \u03b1>0")
print("\u2192 No single \u03b1 stabilizes both.")
print("=" * 72)

# Convergence comparison: optimal vs boundary alpha at a=3.2
print("\n--- Convergence comparison at a=3.2, targeting x*=0.6875 ---")
a_demo = 3.2
x_star_demo = 1 - 1/a_demo
lam_demo = 2 - a_demo
alpha_opt_demo = 1.0 / (1.0 - lam_demo)
alpha_bdy_demo = 2.0 / (1.0 - lam_demo)

x_opt, x_bdy = 0.5, 0.5
print(f"  \u03b1_optimal = {alpha_opt_demo:.4f}, \u03b1_boundary = {alpha_bdy_demo:.4f}")
print(f"  {'iter':>4}  {'x_optimal':>12}  {'x_boundary':>12}  {'err_opt':>10}  {'err_bdy':>10}")
for n in range(12):
    err_o = abs(x_opt - x_star_demo)
    err_b = abs(x_bdy - x_star_demo)
    print(f"  {n:4d}  {x_opt:12.8f}  {x_bdy:12.8f}  {err_o:10.2e}  {err_b:10.2e}")
    x_opt = alpha_opt_demo * logistic(a_demo, x_opt) + (1 - alpha_opt_demo) * x_opt
    x_bdy = alpha_bdy_demo * logistic(a_demo, x_bdy) + (1 - alpha_bdy_demo) * x_bdy

print(f"\nOptimal converges superlinearly (g'=0 \u2192 error squares each step).")
print(f"Boundary oscillates with |g'|=1 \u2014 does NOT converge.")

### 8.3 Bifurcation Diagrams: Original vs. Transformed

In [ ]:
a_range = np.linspace(2.5, 4.0, 400)
x0_samples = np.linspace(0.1, 0.9, 30)
alpha_bif = -1.0

a_orig, x_orig = compute_bifurcation(logistic, a_range, x0_samples,
                                      n_iter=500, n_discard=400)
a_trans, x_trans = compute_bifurcation_transformed(logistic, alpha_bif, a_range,
                                                    x0_samples, n_iter=500, n_discard=400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.scatter(a_orig, x_orig, s=0.1, c='black', alpha=0.5)
ax1.set_xlabel('Parameter $a$'); ax1.set_ylabel('$x$')
ax1.set_title('Original: $f(x) = ax(1-x)$')

ax2.scatter(a_trans, x_trans, s=0.1, c='black', alpha=0.5)
ax2.set_xlabel('Parameter $a$'); ax2.set_ylabel('$x$')
ax2.set_title(rf'Transformed: $g = \alpha f + (1-\alpha)x$, $\alpha = {alpha_bif}$')

plt.tight_layout()
plt.show()

### 8.4 Position-Dependent $\alpha(x)$: Simultaneous Stabilization

Here we demonstrate that $\alpha(x) = 1/(1-f'(a,x))$ overcomes the Proposition 4 obstruction, simultaneously stabilizing both fixed points of the logistic map at $a = 4$.

In [ ]:
import importlib
import petrification.transforms, petrification.iteration
importlib.reload(petrification.transforms)
importlib.reload(petrification.iteration)
from petrification.transforms import make_alpha_func
from petrification.iteration import iterate_transformed

def logistic_prime(a, x):
    return a * (1 - 2*x)

a_test = 4.0
x_fp0 = 0.0
x_fp1 = 1 - 1/a_test  # = 0.75

alpha_func = make_alpha_func(logistic_prime, a_test, smooth=True, cap=5.0)

# Verify: both fixed points get g'(x*) = 0
for x_fp, label in [(x_fp0, 'x*=0'), (x_fp1, 'x*=0.75')]:
    lam = logistic_prime(a_test, x_fp)
    a_val = alpha_func(x_fp)
    g_prime = 1 + a_val * (lam - 1)
    print(f"{label}: f'={lam:+.2f}, \u03b1(x*)={a_val:+.4f}, g'(x*)={g_prime:+.6f}")

# Compare convergence: constant α vs α(x)
alpha_const = 1.0 / (1.0 - logistic_prime(a_test, x_fp1))  # optimal for x*=0.75
n_iter = 60
x0_near = 0.8

traj_none = iterate_transformed(logistic, 1.0, a_test, x0_near, n_iter)
traj_const = iterate_transformed(logistic, alpha_const, a_test, x0_near, n_iter)
traj_func = iterate_transformed(logistic, alpha_func, a_test, x0_near, n_iter)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, traj, title in [
    (axes[0], traj_none, f'No transform (\u03b1=1)'),
    (axes[1], traj_const, f'Constant \u03b1={alpha_const:.4f}'),
    (axes[2], traj_func, '\u03b1(x) = 1/(1-f\'(a,x))'),
]:
    ax.plot(traj, 'b-', lw=0.8, alpha=0.7)
    ax.axhline(x_fp1, color='red', ls='--', alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel('$x_n$')
    ax.set_xlabel('Iteration')
    ax.set_ylim(-0.5, 1.5)

plt.suptitle('Proposition 4 Obstruction Overcome by \u03b1(x)', fontsize=13)
plt.tight_layout()
plt.show()

# Convergence rates across Feigenbaum cascade
a_cascade = np.linspace(2.5, 4.0, 60)
conv_rate_const = np.zeros(len(a_cascade))
conv_rate_func = np.zeros(len(a_cascade))

for j, a_v in enumerate(a_cascade):
    x_fp = 1 - 1/a_v if a_v > 1 else 0
    af = make_alpha_func(logistic_prime, a_v, smooth=True, cap=5.0)
    ac = 1.0 / (1.0 - logistic_prime(a_v, x_fp)) if abs(1 - logistic_prime(a_v, x_fp)) > 0.01 else 1.0
    
    n_conv_c, n_conv_f = 0, 0
    for x0 in np.linspace(0.01, 0.99, 20):
        tc = iterate_transformed(logistic, ac, a_v, x0, 100)
        tf = iterate_transformed(logistic, af, a_v, x0, 100)
        if abs(tc[-1] - x_fp) < 1e-6: n_conv_c += 1
        if abs(tf[-1] - x_fp) < 1e-6: n_conv_f += 1
    conv_rate_const[j] = n_conv_c / 20
    conv_rate_func[j] = n_conv_f / 20

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax.plot(a_cascade, conv_rate_const, 'b-o', ms=4, label='Constant \u03b1 (optimal at x*)')
ax.plot(a_cascade, conv_rate_func, 'r-s', ms=4, label='\u03b1(x) = 1/(1-f\'(a,x))')
ax.axvline(3.0, color='gray', alpha=0.3, ls='--')
ax.axvline(3.449, color='gray', alpha=0.3, ls=':')
ax.axvline(3.5699, color='gray', alpha=0.3, ls='-.')
ax.set_xlabel('Parameter $a$')
ax.set_ylabel('Convergence fraction (20 ICs)')
ax.set_title('\u03b1(x) vs constant \u03b1: local superiority vs global basin fragmentation')
ax.legend()
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

## 9. Application to Quantum Eigenvalue Problems

### 9.1 Eigenstates as Projective Fixed Points

We demonstrate the eigenvalue-fixed-point connection directly: power iteration converges to the dominant eigenstate, and inverse power iteration with different shifts selects different eigenstates — exactly like the alpha-transform selects different fixed points.

In [ ]:
# Harmonic oscillator eigenvalues via matrix diagonalization
N = 500
x_grid = np.linspace(-8, 8, N)

eigenvalues, eigenvectors = solve_eigenstates(harmonic, x_grid, n_states=8)
exact = np.arange(8) + 0.5

print("Harmonic oscillator eigenvalues:")
print(f"{'n':>3}  {'Computed':>12}  {'Exact':>8}  {'Error':>10}")
for i in range(8):
    print(f"{i:3d}  {eigenvalues[i]:12.6f}  {exact[i]:8.1f}  {abs(eigenvalues[i] - exact[i]):10.2e}")

In [ ]:
# Inverse power iteration: convergence to different eigenstates via shift
H = discretize_hamiltonian(harmonic, x_grid)
evals_full = eigh(H, eigvals_only=True)

fig, ax = plt.subplots(figsize=(10, 6))
shifts = [0.4, 1.6, 2.4, 3.6, 4.6]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for sigma, color in zip(shifts, colors):
    rq = inverse_power_iteration(H, sigma, n_iter=40)
    nearest_idx = np.argmin(np.abs(evals_full - sigma))
    nearest_E = evals_full[nearest_idx]
    ax.plot(rq, color=color, lw=2,
            label=rf'$\sigma={sigma:.1f} \to E_{nearest_idx}={nearest_E:.2f}$')

for i in range(5):
    ax.axhline(evals_full[i], color='gray', alpha=0.3, lw=0.8)

ax.set_xlabel('Iteration $n$')
ax.set_ylabel(r'Rayleigh quotient $\langle x|H|x\rangle$')
ax.set_title('Inverse power iteration = fixed-point convergence to nearest eigenstate\n'
             'Each shift selects a different attracting fixed point')
ax.legend(fontsize=10)
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

In [ ]:
# Alpha-scan: which eigenstate does the iteration converge to for each alpha?
N_small = 60
x_small = np.linspace(-5, 5, N_small)
H_small = discretize_hamiltonian(harmonic, x_small)

alphas = np.linspace(-2.0, 2.0, 500)
rng = np.random.default_rng(7)
x0 = rng.standard_normal(N_small)

dom, quality, ev_small = alpha_eigenstate_scan(H_small, x0, alphas, n_iter=500)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(alphas, dom, c=quality, cmap='viridis', s=3, vmin=0, vmax=1)
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('Dominant eigenstate index')
ax.set_title(r'$\alpha$-transform selects different eigenstates (color = convergence quality)')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r'$|\langle\psi_k|x\rangle|^2$')
plt.tight_layout()
plt.show()

## 10. Assessment and Further Considerations

### 10.1 What Is Established

- **Props 1-5** are clean, provable results about the alpha-transform's effect on fixed-point stability
- The **$\alpha(x)$ generalization** overcomes the constant-$\alpha$ obstruction and provides independent local control
- The **projective fixed-point interpretation** of eigenstates is mathematically rigorous (not novel, but useful framing)
- The alpha-transform is a member of the **Krasnoselskii-Mann family** of relaxation methods

### 10.2 What Emerged From Later Investigations

Several findings from the companion notebooks have strengthened or redirected the original ideas:

**1. The Dyson equation universality** ([crossover_alpha_turbiner](crossover_alpha_turbiner.ipynb)):  
Constant $\alpha \approx 0.5$ provides universal convergence for scalar Dyson equations $G = 1/(\omega - \varepsilon_0 - U^2 G)$ across *all* coupling strengths $U = 0.3 \to 3.0$. Naive iteration ($\alpha = 1$) diverges for $U \geq 1.0$, but the half-step relaxation converges to machine precision in ~13 iterations. This is the strongest concrete application of the $\alpha$-transform: **it explains why self-consistent resummation works** in many-body physics. This is likely publishable.

**2. Perturbation detection via $\alpha(x)$ profiles** ([perturbation_detection](perturbation_detection.ipynb), [quantum_perturbation_detection](quantum_perturbation_detection.ipynb)):  
When a physical system deviates from a known model, the measured $\alpha(x)$ profile ($\alpha$ required to map the model's relaxation to the observed relaxation) **exactly encodes** the perturbation:

$$V'_\text{pert}(x) = k_\text{model} \cdot x \cdot (\alpha(x) - 1)$$

This works for classical springs (parallel Hooke, displaced equilibrium, quartic anharmonicity) and quantum systems (fine structure, vacuum polarization, Gaussian wells on hydrogen). The shape of $\alpha(r) - 1$ classifies the perturbation type: power-law ($1/r^n$) perturbations give power-law $\alpha$ profiles; exponential perturbations (Yukawa) give exponential profiles. This is a genuinely novel **diagnostic tool**: the transform serves not as a computational accelerator but as a **probe of unmodeled physics**.

**3. Chaos control as static OGY** ([inverse_correspondence](inverse_correspondence.ipynb)):  
Position-dependent $\alpha(x)$ provides **targeted chaos suppression** analogous to OGY chaos control [Ott, Grebogi, Yorke, PRL 1990], but as a static map modification rather than dynamic feedback. A localized Gaussian $\alpha(x)$ centered on an unstable fixed point $x^*$ with $\alpha(x^*) = 1/(a-1)$ produces **superstable** convergence ($g'(x^*) = 0$), converting the fully chaotic logistic map at $a = 4.0$ ($\Lambda = +0.69$) into a system that converges to the fixed point with machine precision. The key insight: chaotic orbits are ergodic, so they *inevitably* encounter the localized modification and get captured.

**4. Lyapunov exponent relationship under constant $\alpha$** ([inverse_correspondence](inverse_correspondence.ipynb)):  
The relationship between Lyapunov exponents of regular ($\alpha = 1$) and inverted ($\alpha = -1$) logistic maps is an open question explored in §4 of the inverse_correspondence notebook. The transformed derivative $g'(x) = 1 + \alpha(f'(x) - 1)$ modifies the local stretching at every point, but the cumulative effect on the Lyapunov exponent $\Lambda = \lim_{N\to\infty} \frac{1}{N} \sum \log|g'(x_n)|$ is non-trivial because the orbit changes. For $\alpha = -1$ specifically, $g'(x) = 2 - f'(x)$, so the Lyapunov sum involves $\log|2 - f'(x_n)|$ along a *different* trajectory. Whether this preserves, anti-correlates, or non-trivially relates to the original Lyapunov exponent awaits numerical investigation via the $(\alpha, a)$ phase diagram.

### 10.3 What Might Be Worth Developing

1. **The $\alpha'$ cancellation theorem** (that $g'(x^*)$ depends only on $\alpha(x^*)$, not $\alpha'(x^*)$) is implicit in the preconditioning literature but rarely stated this cleanly. Worth formalizing. *Update*: this result is the foundation of the chaos control application (item 3 above) — not just a curiosity but the mechanism enabling localized map modification.

2. **The concavity-inversion perspective** ($g'' = \alpha f''$, so $\alpha < 0$ flips curvature) may have applications beyond iteration — e.g., in optimization where convexity/concavity of the objective matters.

3. **Basin fragmentation from $\alpha(x)$**: The singularity at $f'(a,x) = 1$ is where $\alpha(x) \to \infty$. Regularizing this (e.g., $\alpha(x) = 1/(1-f'(a,x) + \epsilon)$) might preserve the local superlinear convergence while maintaining global basin connectivity.

4. **Matrix-valued $\alpha$**: For multi-dimensional fixed-point problems ($\mathbf{x} = \mathbf{f}(\mathbf{x})$), $\alpha$ becomes a matrix. The optimal $\alpha = (I - J_f(x^*))^{-1}$ is exactly the Newton step direction. This connects the alpha-transform to Newton's method in a precise way. *Update*: the Dyson equation results suggest this direction is promising — matrix Dyson equations in DMFT are the natural next target.

5. ~~Application to Dyson equations — see crossover notebook.~~ **Done.** See [crossover_alpha_turbiner](crossover_alpha_turbiner.ipynb). The key result ($\alpha^* \approx 0.5$ universality) exceeded expectations.

### 10.4 Revised Assessment of Novelty

The original assessment was conservative: "likely not a new discovery." After the companion investigations, the picture is more nuanced:

| Aspect | Novel? | Status |
|--------|--------|--------|
| Core transform $g = \alpha f + (1-\alpha)x$ | No — Krasnoselskii-Mann, SOR | Well-known |
| Props 1-5 (stability theorems) | Partially — clean synthesis | Pedagogical contribution |
| $\alpha'$ cancellation at fixed points | Implicit in literature | Clean formalization |
| $\alpha(x)$ for simultaneous stabilization | Likely implicit | Clean formalization |
| $\alpha^* \approx 0.5$ for Dyson equations | **Yes** | Publishable (empirical $\lambda = 0.5$ used in GW codes but never derived) |
| $\alpha(x)$ as perturbation diagnostic | **Yes** | Novel tool (no prior art found) |
| Static chaos control via $\alpha(x)$ | New framing of OGY | Novel application |
| Lyapunov exponent relationship under $\alpha$ | Unknown | Open question — needs numerical investigation |
| Gauge-theoretic framing | Suggestive | Not yet productive |

### 10.5 Prior Art Check

**Literature search conducted April 2026** (Google Scholar). Results:

- [ ] Does the preconditioning literature already state the $\alpha'$ cancellation for position-dependent relaxation?
  - *Searched: "position-dependent relaxation" + "fixed point", "Krasnoselskii-Mann" + "state-dependent relaxation". **No results found.** The cancellation appears implicit in abstract fixed-point theory but is not stated in this explicit form.*
- [ ] Is the "concavity inversion" framing ($g'' = \alpha f''$) used anywhere in optimization theory?
  - *Not searched yet.*
- [ ] Are there results connecting gauge transformations on iteration parameters to fixed-point stability?
  - *Not searched yet.*
- [ ] Is the basin fragmentation from singular $\alpha(x)$ studied in the dynamical systems literature?
  - *Not searched yet.*
- [x] Has anyone applied position-dependent relaxation to Dyson equations specifically?
  - *Constant mixing parameters are standard in self-consistent GW/DMFT codes: Chibani et al. (PRB 2016) use $\lambda = 0.5$; Caruso et al. (PRB 2013) and Forster (JCTC 2025) use adaptive mixing. Tadano & Tsuneyuki (PRB 2015) use $\alpha = 0.1$ for phonon self-consistency. **However, no derivation of why specific values work exists.** All are tuned empirically. The crossover notebook's analytical derivation of $\alpha^* \approx 0.5$ from the fixed-point stability condition appears novel.*
  - *Essl et al. (arXiv:2502.01420, 2025), "How to stay on the physical branch," analyzes which fixed point mixing converges to — closest related work but addresses a different question (branch selection vs. optimal mixing rate).*
- [x] Is the $\alpha(x)$ perturbation detection formula known in the control theory or inverse problems literature?
  - *Searched: "relaxation parameter" + "perturbation detection" + "inverse problem", "position dependent" + "relaxation" + "perturbation reconstruction" + "fixed point". **Zero results.** The use of a variable relaxation profile as a diagnostic tool for identifying unknown perturbations appears genuinely novel.*
- [x] Is the "Krasnoselskii-Mann with state-dependent parameter" framework studied?
  - *Abstract convergence theory exists (variable step sizes for nonexpansive operators), but always for convergence acceleration. **No results using the relaxation profile itself as a diagnostic/measurement tool.***